# Sub Shader
Sub Shader is a real-time audio visualizer.

# Overview
The goal is to create a highly accurate and responsive audio analysis tool that depicts what the audio sounds like. The result is an audio-graphics pipeline that monitors audio in real time by analyzing and converting it into a visual representation. This project brings together my interests in digital signal processing, parallel computing, real-time systems, and graphics programming. It's a deep dive into high-performance audio visualization using advanced DSP methods and GPU acceleration. 

To accomplish this, I've obtained a deep understanding of signal processing and parallel computing through extensive research into various papers, tutorials, and implementations. I now feel enabled to generalize the fundamental concepts in DSP into various contexts, not just audio, for pattern detection and feature extraction. Profiling and optimizing the performance bottlenecks has taught me how to identify the independent operations in a pipeline and balance the gains from  hardware acceleration against the cost of distributing the workload across computational resources.

# Performance Analysis
This section depicts the performance of Subshader at its current standing. 

## Dynamic Plotting
TODO Insert gif
This is example audio being visualized at TODO FPS. 

## Plot Comparison
TODO Insert example MIDI and Audio Time Series 

TODO Insert PyWavelet vs Subshader performance times and side by side the plots

PyWavelet is a popular wavelet library. Here is a side by of the plot.

## Timing Comparison
TODO Do timing analysis on PyWavelet vs CuWavelet

TODO Inform how many points are being processed. Insert timing analysis in a matrix where the first row is the audio time benchmark, the second row is the pywavelet and subshader wavelet timings, and the third row is the plotting update (maybe do matplot lib if its easy). 

# Design Goals
To acheive this, the overall processing needs to be accurate and fast enough to keep up with the audio playback. 

# Flowchart
![SubShader Main Loop Flowchart](assets/diagrams/subshader_main_loop.drawio.png)

# Software Modules

## Audio
The Audio module delivers raw audio samples to the DSP module and implements a simple overlapping window scheme. On init, it determines the audio's metadata (sample rate, mono vs stereo, etc) and during runtime retrieves a consistent number of audio samples each call. The overlap mechanism ensures that each audio window overlaps a portion of its samples, smoothing continuity in the time axis by reducing artifacts at the window's edges.

TODO Insert example of audio overlap

Currently, the input stream of audio data comes from file IO, but will soon support a live source of audio.

## DSP
The purpose of the DSP module is to analyze the input audio signal for its time-frequency content - want to know when in time which frequencies are present. The Continuous Wavelet Transform (CWT) is used to convert the input audio from the time domain and transform it to the time-frequency domain, which allows us to see which frequencies are present at specific points in time. The result is a 2D scalogram where the value of every point at each time-frequency coordinate is the relative strength of that frequency's activity at that particular time.

After the computing the CWT, the results are normalized to account for energy bias introduced in the CWT. A small portion of the CWT results are discarded to reduce edge effects produced by the CWT and the overlapping audio scheme. Finally the output is downsampled to help performance, reducing the total number of samples being handled. 

To read an in-depth explanation of the CWT and its specific design decisions, click here: TODO Insert link

## Plot
The Plot is a 2D grid with two axes: frequency vs time. It displays the chronologically-ordered CWT results in a continuous reel. The values of each point are mapped to a color spectrum, visualizing the relative strength of each time-frequency coordinate. 

Real-time plotting is a little challenging, mainly because of the large quantity of points that need to be displayed and updated quickly. Most Python plotting libraries struggle to render very large quantities of points in real-time. To help speed things up, a GPU-based shader is used to plot the 2D data efficiently using graphics hardware. This alleviates a huge performance bottleneck in the pipeline but soon there will be an alternative method of plotting that is GPU-independent.

# Benchmark
Here is where we discuss the performance of Sub Shader

# Near Future Improvements

At the moment, the current visualization is not synchronized to the audio playback, so technically its performance is not considered real-time. Because its update rate is nondeterministic, it just pushes through the pipeline as fast as it can, and the python can only run single threaded, it doesn't have a deterministic behavior, and can lag depending on what the system's load is at the moment, it's not real time. It doesn't update according to deadlines, it updates as fast as it can. The goal in the near future is to synchronize the visualization to the audio by tapping into my systems soundcard interface (going to beef up audio input).

The other near future improvement is to implement a ring buffer in the GPU to only update the plot with new data instead of entire continuous chronological history of the signal, greatly reducing CPU to GPU transfer sizes. This is a little tricky when the overlap changes. Will also be enforcing a configs param check that ensures the audio window, overlap and downsample can all slide together

